# INFO-6147-(01)-26W Deep Learning with Pytorch

## Project:

**Student Name:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 30th, 2026

**Description:**

## Setup Env

In [ ]:
!pip install datasets
!pip install gradio
!pip install streamlit
!pip install pyngrok

## Import Libs

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from sklearn.metrics import confusion_matrix, f1_score
import seaborn as sns
from torch.utils.data import random_split,Subset
from tqdm import tqdm
import streamlit as st
from groq import Groq
import io

## Check Device

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
EPOCHS=5
BATCH_SIZE = 128
LEARNING_RATE=0.001
DATA_PATH = './data'

## Load Dataset (Colab)

In [ ]:
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset
train_dataset = datasets.Food101(root=DATA_PATH, split='train', download=True, transform=transform)
test_dataset = datasets.Food101(root=DATA_PATH, split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

num_classes = len(train_dataset.classes)
print(f"Load Food101 Successfully！Num of Classes: {num_classes}")

## Build the model

In [ ]:
# model = models.resnet50(pretrained=True)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)

## Train the model

In [ ]:
history={
    'train_loss': [], 'train_acc': [],
    'test_loss': [], 'test_acc': []
}

for epoch in range(EPOCHS):
    # --- train process ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    # training bar
    train_pbar = tqdm(enumerate(train_loader), total=len(train_loader), desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for i, (images, labels) in train_pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        #calculate
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        # update process bar
        train_pbar.set_postfix({'Loss': f"{loss.item():.4f}", 'Acc': f"{100.*correct/total:.2f}%"})

    epoch_train_loss = running_loss / len(train_loader)
    epoch_train_acc = 100. * correct / total

    # --- (Evaluate) ---
    model.eval()
    t_loss, t_correct, t_total = 0.0, 0, 0


    test_pbar = tqdm(test_loader, total=len(test_loader), desc=f"Epoch {epoch+1}/{EPOCHS} [Test]", leave=False)

    with torch.no_grad():
        for images, labels in test_pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            t_loss += loss.item()
            _, predicted = outputs.max(1)
            t_total += labels.size(0)
            t_correct += predicted.eq(labels).sum().item()

    epoch_test_loss = t_loss / len(test_loader)
    epoch_test_acc = 100. * t_correct / t_total

    # record to history
    history['train_loss'].append(epoch_train_loss)
    history['train_acc'].append(epoch_train_acc)
    history['test_loss'].append(epoch_test_loss)
    history['test_acc'].append(epoch_test_acc)

    print(f"==> Epoch {epoch+1} ends | Train Acc: {epoch_train_acc:.2f}% | Test Acc: {epoch_test_acc:.2f}%")

## Graphing Loss Curve and Accuarcy Curve

In [ ]:
plt.figure(figsize=(12, 5))

# Loss curve
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS+1), history['train_loss'], label='Train Loss', marker='o')
plt.plot(range(1, EPOCHS+1), history['test_loss'], label='Test Loss', marker='o')
plt.title('Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

# Accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS+1), history['train_acc'], label='Train Acc', marker='s')
plt.plot(range(1, EPOCHS+1), history['test_acc'], label='Test Acc', marker='s')
plt.title('Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.show()

## Visualize result

In [ ]:
def visualize_results(model, dataset, num_images=6):
    model.eval()
    fig = plt.figure(figsize=(15, 10))
    class_names = dataset.classes

    # select randomly
    indices = np.random.choice(len(dataset), num_images, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, label = dataset[idx]
            input_tensor = image.unsqueeze(0).to(device)

            output = model(input_tensor)
            _, pred = torch.max(output, 1)

            # transfer into showable picture
            img_display = image.permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_display = std * img_display + mean
            img_display = np.clip(img_display, 0, 1)

            ax = plt.subplot(2, 3, i + 1)
            color = 'green' if pred.item() == label else 'red'
            ax.set_title(f"Pred: {class_names[pred.item()]}\nActual: {class_names[label]}", color=color)
            plt.imshow(img_display)
            plt.axis('off')
    plt.show()

# visualize
visualize_results(model, test_dataset)

## Setup Enviroment for Llama

In [79]:
!pip install streamlit groq pyngrok

## API

In [ ]:
from google.colab import userdata

# --- Get API key via Colab Secrets ---
try:

    api_key_from_colab = userdata.get("GROQ_API_KEY")
    print(api_key_from_colab)
except:
    api_key_from_colab = None
    print("❌ Please enter your API Key in the sidebar!")

## Create UI

In [82]:
%%writefile app.py
import streamlit as st
from groq import Groq
from google.colab import userdata

# --- Get API key via Colab Secrets ---
try:

    api_key_from_colab = userdata.get("GROQ_API_KEY")
    print(api_key_from_colab)
except:
    api_key_from_colab = None
    print("❌ Please enter your API Key in the sidebar!")


# --- 1. Basic Configuration ---
st.set_page_config(page_title="AI Food Guide", page_icon="🍣")

# Sidebar for API Key input to ensure privacy
with st.sidebar:
    st.title("🛠️ Settings")
    # set colab key as default, otherwise set api key as None

    api_key = st.text_input("Enter Groq API Key",
                            value=api_key_from_colab if api_key_from_colab else "",
                            type="password")
    print(api_key)
    if not api_key:
        st.info("Get your key at [console.groq.com](https://console.groq.com/)")



# --- 2. Llama Core Function ---
def ask_llama_chef(food_name, api_key):
    if not api_key:
        return "❌ Please enter your API Key in the sidebar!"

    try:
        client = Groq(api_key=api_key)
        prompt = f"""
        You are an expert Japanese travel guide serving a tourist from Canada.
        The vision model has identified this dish as "{food_name}".
        Please provide information in English: 1. Taste & Texture, 2. Travel Trivia, 3. Advice for Travelers.
        Tone: Humorous and friendly. Keep it under 150 words.
        """
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error occurred: {str(e)}"

# --- 3. UI Interface ---
st.title("🍣 AI Japanese Food Guide")
mock_predicted_label = st.selectbox(
    "Select a recognition result to test:",
    ["Sushi", "Ramen", "Takoyaki", "Tempura", "Okonomiyaki"]
)

if st.button("View Food Guide"):
    with st.spinner("Thinking..."):
        guide_text = ask_llama_chef(mock_predicted_label, api_key)
        st.chat_message("assistant", avatar="👨‍🍳").write(guide_text)

Overwriting app.py


## Setup for Streamlit
* localtunnel
* Get Tunnel IP
* Activate Streamlit and turn it on

In [83]:
# !pip install streamlit groq
# !npm install -g localtunnel

# install Python module
!pip install streamlit groq
# dlownload Cloudflare tunnle tool
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

--2026-04-04 21:56:13--  https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
Resolving github.com (github.com)... 140.82.114.3
Connecting to github.com (github.com)|140.82.114.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb [following]
--2026-04-04 21:56:13--  https://github.com/cloudflare/cloudflared/releases/download/2026.3.0/cloudflared-linux-amd64.deb
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/106867604/ec689fe1-d727-4ebd-bbc3-5967730ab54e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-04-04T22%3A40%3A04Z&rscd=attachment%3B+filename%3Dcloudflared-linux-amd64.deb&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&

In [84]:
!curl ipv4.icanhazip.com

35.243.161.190


In [85]:
# !streamlit run app.py & npx localtunnel --port 8501

import os
import subprocess
import time

# 1. 確保安裝了 cloudflared (如果剛才裝過可以跳過這兩行)
if not os.path.exists("cloudflared-linux-amd64.deb"):
    !wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb

# 2. 啟動 Streamlit (在背景執行)
os.system("streamlit run app.py &")

# 3. 啟動 Cloudflare 隧道並擷取輸出網址
print("正在建立安全隧道，請稍候...")
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# 4. 從 Log 中找尋 trycloudflare.com 的網址
for line in p.stdout:
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "="*30)
        print(f"✅ 你的網頁已準備就緒！")
        print(f"🔗 點擊進入： {url}")
        print("="*30)
        break

正在建立安全隧道，請稍候...

✅ 你的網頁已準備就緒！
🔗 點擊進入： trycloudflare.com...


## Activate Tunnel

In [86]:
import os
import subprocess
import time

# 1. activate Streamlit (background)
os.system("streamlit run app.py &")

# Activate Clounflare and get  the url
print("⏳Creating the connection...")
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Find trycloudflare.com link
for line in p.stdout:
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "★" * 40)
        print(f"Web is ready")
        print(f"Click the url to open the webpage： {url}")
        print("★" * 40)
        break

⏳Creating the connection...

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Web is ready
Click the url to open the webpage： trycloudflare.com...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★


In [87]:
import subprocess
import time
import socket

# --- 1. Check if Streamlit is already running on port 8501 ---
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# --- 2. Start Streamlit if it's not already running ---
if not is_port_open(8501):
    print("🚀 Starting Streamlit in the background...")
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5) # Give it time to boot
else:
    print("✅ Streamlit is already running.")

# --- 3. Start Cloudflare Tunnel and FORCE print logs ---
print("🌐 Opening Cloudflare Tunnel... (Look for the '.trycloudflare.com' link below)")
print("-" * 50)

# We use stdbuf to disable buffering so the URL appears immediately
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# This loop will print EVERYTHING Cloudflare says until the URL appears
for line in p.stdout:
    print(f"DEBUG: {line.strip()}") # This helps us see if there's an error
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "★" * 50)
        print(f"🔥 SUCCESS! YOUR APP IS LIVE AT:")
        print(f"👉 {url}")
        print("★" * 50)
        # We don't break, so the tunnel stays active in this cell

✅ Streamlit is already running.
🌐 Opening Cloudflare Tunnel... (Look for the '.trycloudflare.com' link below)
--------------------------------------------------
DEBUG: 2026-04-04T21:56:23Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
DEBUG: 2026-04-04T21:56:23Z INF Requesting new quick Tunnel on trycloudflare.com...

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🔥 SUCCESS! YOUR APP IS LIVE AT:
👉 trycloudflare.com...
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

KeyboardInterrupt: 